# 02 — GRPO training (Unsloth Qwen2.5-1.5B-Instruct + TRL)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LonelyGuy-SE1/cybersec_env/blob/main/notebooks/02_grpo_train.ipynb)

Train a small Qwen defender on the cybersec env using GRPO with **six independent reward functions**:

1. `reward_json_valid` — does the model emit valid JSON?
2. `reward_schema_valid` — does it parse into a `CybersecAction`?
3. `reward_target_in_valid_targets` — is the chosen target legal?
4. `reward_no_redundant_containment` — penalise re-isolating already-isolated assets.
5. `reward_step_total` — actual step reward from running the action against a *cloned* env snapshot.
6. `reward_avoids_exfil_path` — bonus for picking actions on attack-path targets when alerts are present.

## Fast-first-run workflow

The first cell flips a single `FAST_DEV_MODE` knob:

| Mode | Dataset | GRPO steps | Generations / step | Wall clock on Colab T4 |
|------|---------|------------|--------------------|------------------------|
| `FAST_DEV_MODE = True`  *(default)* | ~64 prompts | 8  | 2 | **~8–12 min** |
| `FAST_DEV_MODE = False` *(full run)* | ~2000 prompts | 200 | 4 | ~90–120 min |

Run it once with the default to confirm everything works end-to-end. Once the
fast run finishes cleanly, flip the flag and rerun for the full training.

Outputs:
* `_artifacts/qwen_cybersec_lora/` — the trained LoRA adapter
* `_artifacts/training_log.json` — per-step reward components

## 0. Install

In [ ]:
%%capture
# Colab-friendly install. When this notebook is opened on Colab the repo
# isn't cloned automatically; we detect that case and clone first. Locally
# (when the repo is already on disk) we just run pip install against the
# checkout.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LonelyGuy-SE1/cybersec_env.git"
REPO_DIR = Path("/content/cybersec_env") if "google.colab" in sys.modules else Path("..").resolve()

if "google.colab" in sys.modules and not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

PKG_DIR = REPO_DIR / "cybersec"
assert PKG_DIR.exists(), f"cybersec package not found at {PKG_DIR}"

# Pinned for the free Colab T4 (CUDA 12.x, py 3.10/3.11). Unsloth pulls in
# its own torch build so we install it before TRL/peft/accelerate.
!pip install -q -e {PKG_DIR}
!pip install -q "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes
!pip install -q matplotlib pandas

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
from pathlib import Path

import torch
from datasets import Dataset

from cybersec import (
    ActionType,
    CybersecAction,
    CybersecEnvironment,
    list_scenarios,
)
from cybersec.baselines import HeuristicPolicy, RandomPolicy

# ---------------------------------------------------------------------------
# Smoke vs full run.
#
# FAST_DEV_MODE = True  -> ~8-12 min on a Colab T4. Use this for the very
#                          first run end-to-end so you can confirm the env,
#                          dataset, reward funcs, Unsloth load and GRPO loop
#                          all work together before committing to a 90 min
#                          training run.
# FAST_DEV_MODE = False -> full ~90-120 min training run.
#
# Everything below this block keys off `MODE` -- nothing else needs to
# change between runs.
# ---------------------------------------------------------------------------
FAST_DEV_MODE = True

if FAST_DEV_MODE:
    MODE = {
        "label":             "fast_dev",
        "seeds_per_scenario": 4,
        "max_rows":           64,
        "max_steps":          8,
        "num_generations":    2,
        "per_device_bs":      1,
        "grad_accum":         2,
        "logging_steps":      1,
        "save_steps":         8,
    }
else:
    MODE = {
        "label":             "full",
        "seeds_per_scenario": 40,
        "max_rows":           2000,
        "max_steps":          200,
        "num_generations":    4,
        "per_device_bs":      2,
        "grad_accum":         4,
        "logging_steps":      5,
        "save_steps":         100,
    }

print(f"[mode] {MODE['label']}: max_steps={MODE['max_steps']}  "
      f"num_generations={MODE['num_generations']}  max_rows={MODE['max_rows']}")

ARTIFACTS = Path("_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
ADAPTER_DIR = ARTIFACTS / f"qwen_cybersec_lora_{MODE['label']}"
TRAIN_LOG = ARTIFACTS / f"training_log_{MODE['label']}.json"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SCENARIOS = list_scenarios()
MAX_PROMPT_LEN = 1024
MAX_NEW_TOKENS = 48
SEEDS_FOR_DATASET = list(range(MODE["seeds_per_scenario"]))

## 1. Build the state-snapshot dataset

GRPO needs a fixed corpus of *prompts*. We follow heuristic rollouts and at
every step snapshot the env (deep copy of the live `CybersecEnvironment`)
alongside the rendered prompt. At training time, GRPO will sample K candidate
actions from the model for each prompt and score them by *cloning* the
snapshot and running `env.step(candidate)`.

We cap at ~2k examples — enough for 1-2 GRPO epochs on a T4.

In [ ]:
SYSTEM_PROMPT = (
    "You are an SRE-grade cyber-defender driving an OpenEnv environment.\n"
    "Reply with exactly one JSON object on one line of the form\n"
    '{"action_type": "...", "target": "..."}.\n'
    "action_type must be one of MONITOR, INVESTIGATE, ISOLATE_ASSET, REVOKE_IDENTITY, BLOCK_EGRESS, PATCH_ASSET.\n"
    "target must be omitted (or null) for MONITOR; otherwise it must come from valid_targets."
)

def render_observation(obs) -> str:
    lines = [
        f"tick={obs.tick}/{obs.horizon}  scenario={obs.scenario_id}  attacker={obs.attacker_personality.value}",
        f"isolated_assets={obs.isolated_assets}",
        f"revoked_identities={obs.revoked_identities}",
        f"blocked_egress={obs.blocked_egress_assets}",
        f"patched={obs.patched_assets}",
        f"confirmed_compromised={obs.confirmed_compromised}",
        f"valid_targets={obs.valid_targets}",
        "recent_alerts:",
    ]
    for a in obs.alerts[-6:]:
        lines.append(f"  t={a.tick} {a.signal.value} sev={a.severity} asset={a.asset} id={a.identity} :: {a.description}")
    lines.append("recent_forensics:")
    for f in obs.forensics[-4:]:
        lines.append(f"  t={f.tick} {f.target_kind}={f.target} compromised={f.is_compromised} conf={f.confidence}")
    return "\n".join(lines)

import pickle, base64

def snapshot_env(env: CybersecEnvironment) -> str:
    """Pickle + base64 the live env so GRPO reward fns can clone it cheaply."""
    return base64.b64encode(pickle.dumps(env)).decode("ascii")

def restore_env(blob: str) -> CybersecEnvironment:
    return pickle.loads(base64.b64decode(blob.encode("ascii")))

rows = []
for scenario_id in SCENARIOS:
    for seed in SEEDS_FOR_DATASET:
        env = CybersecEnvironment()
        policy = HeuristicPolicy()
        obs = env.reset(seed=seed, scenario_id=scenario_id)
        while not obs.done:
            blob = snapshot_env(env)
            prompt = render_observation(obs)
            rows.append({
                "prompt": prompt,
                "system": SYSTEM_PROMPT,
                "valid_assets": obs.valid_targets["assets"],
                "valid_identities": obs.valid_targets["identities"],
                "isolated_assets": list(obs.isolated_assets),
                "revoked_identities": list(obs.revoked_identities),
                "blocked_egress": list(obs.blocked_egress_assets),
                "patched": list(obs.patched_assets),
                "alert_count": len(obs.alerts),
                "env_snapshot": blob,
            })
            obs = env.step(policy.act(obs))

random.Random(0).shuffle(rows)
rows = rows[: MODE["max_rows"]]

ds = Dataset.from_list(rows)
print("dataset size:", len(ds))
print(ds[0]["prompt"][:300])

## 2. Reward functions

In [ ]:
def _parse_first_json_object(text: str):
    """Very forgiving JSON extractor — finds the first balanced { ... }."""
    text = text.strip()
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start : i + 1])
                except json.JSONDecodeError:
                    return None
    return None

def reward_json_valid(prompts, completions, **kw):
    return [1.0 if _parse_first_json_object(c) is not None else 0.0 for c in completions]

def _parsed_action(c):
    payload = _parse_first_json_object(c)
    if not payload:
        return None
    try:
        return CybersecAction(**payload)
    except Exception:
        return None

def reward_schema_valid(prompts, completions, **kw):
    return [1.0 if _parsed_action(c) is not None else 0.0 for c in completions]

def reward_target_in_valid_targets(prompts, completions, valid_assets=None, valid_identities=None, **kw):
    valid_assets = valid_assets or [None] * len(completions)
    valid_identities = valid_identities or [None] * len(completions)
    out = []
    for c, va, vi in zip(completions, valid_assets, valid_identities):
        action = _parsed_action(c)
        if action is None:
            out.append(0.0); continue
        if action.action_type is ActionType.MONITOR:
            out.append(1.0); continue
        if action.action_type is ActionType.REVOKE_IDENTITY:
            out.append(1.0 if action.target in (vi or []) else 0.0)
        elif action.action_type is ActionType.INVESTIGATE:
            out.append(1.0 if action.target in ((va or []) + (vi or [])) else 0.0)
        else:
            out.append(1.0 if action.target in (va or []) else 0.0)
    return out

def reward_no_redundant_containment(prompts, completions, isolated_assets=None, revoked_identities=None, blocked_egress=None, patched=None, **kw):
    isolated_assets = isolated_assets or [[] for _ in completions]
    revoked_identities = revoked_identities or [[] for _ in completions]
    blocked_egress = blocked_egress or [[] for _ in completions]
    patched = patched or [[] for _ in completions]
    out = []
    for c, iso, rev, blk, pat in zip(completions, isolated_assets, revoked_identities, blocked_egress, patched):
        action = _parsed_action(c)
        if action is None:
            out.append(0.0); continue
        if action.action_type is ActionType.MONITOR:
            out.append(0.5); continue  # neutral
        already = {
            ActionType.ISOLATE_ASSET: iso,
            ActionType.REVOKE_IDENTITY: rev,
            ActionType.BLOCK_EGRESS: blk,
            ActionType.PATCH_ASSET: pat,
        }.get(action.action_type, [])
        out.append(0.0 if action.target in already else 1.0)
    return out

def reward_step_total(prompts, completions, env_snapshot=None, **kw):
    """Clone the env, apply the candidate action, return (clipped) step reward."""
    snapshots = env_snapshot or [None] * len(completions)
    out = []
    for c, blob in zip(completions, snapshots):
        action = _parsed_action(c)
        if action is None or blob is None:
            out.append(-1.0); continue
        try:
            env = restore_env(blob)
            obs = env.step(action)
            r = float(obs.reward or 0.0)
        except Exception:
            r = -1.0
        # Clip into [-2, 2] so the magnitude doesn't drown the structural rewards.
        out.append(max(-2.0, min(2.0, r)))
    return out

def reward_avoids_exfil_path(prompts, completions, valid_assets=None, alert_count=None, **kw):
    """+0.5 if defender takes a containment-class action while alerts are firing."""
    valid_assets = valid_assets or [[] for _ in completions]
    alert_count = alert_count or [0] * len(completions)
    containment_types = {ActionType.ISOLATE_ASSET, ActionType.BLOCK_EGRESS, ActionType.REVOKE_IDENTITY}
    out = []
    for c, n_alerts in zip(completions, alert_count):
        action = _parsed_action(c)
        if action is None:
            out.append(0.0); continue
        if n_alerts > 0 and action.action_type in containment_types:
            out.append(0.5)
        elif n_alerts > 0 and action.action_type is ActionType.INVESTIGATE:
            out.append(0.3)
        else:
            out.append(0.0)
    return out

REWARD_FUNCS = [
    reward_json_valid,
    reward_schema_valid,
    reward_target_in_valid_targets,
    reward_no_redundant_containment,
    reward_step_total,
    reward_avoids_exfil_path,
]
REWARD_NAMES = [f.__name__ for f in REWARD_FUNCS]
REWARD_NAMES

## 3. Load Qwen2.5-1.5B with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_PROMPT_LEN + MAX_NEW_TOKENS,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=0,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model.print_trainable_parameters()

## 4. Format dataset with chat template

In [ ]:
def to_chat_prompt(row):
    msgs = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["prompt"]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return {"prompt": text}

ds_chat = ds.map(to_chat_prompt)
print(ds_chat[0]["prompt"][:500])

## 5. GRPO trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(ARTIFACTS / f"grpo_checkpoints_{MODE['label']}"),
    learning_rate=5e-6,
    per_device_train_batch_size=MODE["per_device_bs"],
    gradient_accumulation_steps=MODE["grad_accum"],
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_NEW_TOKENS,
    num_generations=MODE["num_generations"],
    num_train_epochs=1,
    max_steps=MODE["max_steps"],
    logging_steps=MODE["logging_steps"],
    save_steps=MODE["save_steps"],
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    report_to=[],
    seed=0,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=ds_chat,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
)
trainer.train()

## 6. Save the LoRA adapter

In [ ]:
ADAPTER_DIR.mkdir(exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapter to", ADAPTER_DIR)

## 7. Persist a slimmed training log

TRL writes a full HF `Trainer` log to `output_dir/runs/`. We extract just the
per-step reward components into a small JSON for plotting in notebook 03.

In [ ]:
history = getattr(trainer.state, "log_history", []) or []
TRAIN_LOG.write_text(json.dumps({
    "reward_names": REWARD_NAMES,
    "history": history,
    "model_name": MODEL_NAME,
    "mode": MODE,
}, indent=2))
print("wrote", TRAIN_LOG, "with", len(history), "log rows")